# Cell-Akamai · concurrency evidence replay

**Human/machine-run Jupyter companion.** This notebook verifies the frozen ECA input hash and reads locally committed concurrency summaries. It does not run models, change an Akamai deployment, score the oracle or claim training. Without run receipts it displays `NOT_TESTED`. Results are a nonconfirmatory pilot until independently scored.

## 1. Dataset contract / source identity

In [1]:
from pathlib import Path
import json, hashlib, sys
ROOT=Path.cwd()
if not (ROOT / "dataset/eca_pilot_manifest_v1.json").exists(): ROOT=ROOT.parent
assert (ROOT / "dataset/eca_pilot_manifest_v1.json").exists(), "Start from repository root/notebooks"
manifest=json.loads((ROOT / "dataset/eca_pilot_manifest_v1.json").read_text())
b=(ROOT/manifest["path"]).read_bytes()
assert "sha256:"+hashlib.sha256(b).hexdigest()==manifest["sha256"]
items=[json.loads(line) for line in b.splitlines() if line]
assert len(items)==32 and len(set(r["item_id"] for r in items))==32
print("DATASET_SHA_VERIFIED", manifest["sha256"], "ITEMS",len(items), "MODE", manifest["status"])

DATASET_SHA_VERIFIED sha256:69ced0ca1cf77a00446e2b3db03564eb487b2f2b15e68c34590ee8e96b07c274 ITEMS 32 MODE PILOT_NONCONFIRMATORY


## 2. Load append-only concurrent summary receipts (no model requests)

In [2]:
import importlib.util
spec=importlib.util.spec_from_file_location("dash", ROOT/"tools/concurrency/serve_dashboard.py")
dash=importlib.util.module_from_spec(spec); spec.loader.exec_module(dash)
raw=dash.load_sanitized(ROOT/"runs/concurrency")
rows=raw["rows"]
print("VALIDATED SUMMARY ROWS AVAILABLE",len(rows))
if not rows: print("NOT_TESTED — no real concurrency model receipts in this notebook environment")
else:
 for r in rows[:12]:
  print(r["backend"],r["stage"],r["arm"],"c=",r["concurrency"],"tok/s=",r["aggregate_completion_tokens_per_s"],"p95ms=",r["latency_p95_ms"])

VALIDATED SUMMARY ROWS AVAILABLE 0
NOT_TESTED — no real concurrency model receipts in this notebook environment


## 3. Throughput versus concurrency

Aggregate completion tokens divided by the cell-level wall clock duration, **not** stream chunk count. Streamed inter-request tokens/s² cannot be reliably inferred for overlapping requests.

In [3]:
if rows:
 import matplotlib.pyplot as plt
 fig,ax=plt.subplots(figsize=(9,5))
 groups=sorted(set((r["backend"],r["stage"],r["arm"]) for r in rows))
 for group in groups:
  rr=sorted((r for r in rows if (r["backend"],r["stage"],r["arm"])==group and isinstance(r["aggregate_completion_tokens_per_s"],(int,float))),key=lambda x:x["concurrency"])
  if rr:ax.plot([r["concurrency"] for r in rr],[r["aggregate_completion_tokens_per_s"] for r in rr],marker="o",label="/".join(group))
 ax.set(xlabel="Concurrent ECA items",ylabel="Aggregate completion tokens/s",title="Measured pilot throughput (backend/config/arm separate)")
 ax.legend(fontsize=7);ax.grid(alpha=0.3);plt.show()
else: print("NOT_TESTED — throughput chart intentionally omitted")

NOT_TESTED — throughput chart intentionally omitted


## 4. Latency tradeoffs and scientific claim ceiling

In [4]:
if rows:
 import matplotlib.pyplot as plt
 fig,ax=plt.subplots(figsize=(9,5))
 for group in groups:
  rr=sorted((r for r in rows if (r["backend"],r["stage"],r["arm"])==group and isinstance(r["latency_p95_ms"],(int,float))),key=lambda x:x["concurrency"])
  if rr:ax.plot([r["concurrency"] for r in rr],[r["latency_p95_ms"] for r in rr],marker="o",label="/".join(group))
 ax.set(xlabel="Concurrent ECA items",ylabel="Per-item latency p95 (ms)",title="Pilot latency versus concurrency")
 ax.legend(fontsize=7);ax.grid(alpha=0.3);plt.show()
else: print("NOT_TESTED — latency chart intentionally omitted")
print("ANTICUBE NOT_COMPUTED | ΔG* NOT_COMPUTED | SIGNATURE NOT_SIGNED | MMR NOT_COMMITTED")

NOT_TESTED — latency chart intentionally omitted
ANTICUBE NOT_COMPUTED | ΔG* NOT_COMPUTED | SIGNATURE NOT_SIGNED | MMR NOT_COMMITTED


## 5. Next actions

Run `tools/concurrency/run_concurrency.py` with `--execute` against a verified authorized endpoint. Preserve `cXX_paired_for_scoring` SHA commitments. Claude independently scores them on the held-out oracle **after** answer commitment, recording exact-match, paired discordance and tokens per correct item. Never infer DAISY weight training from inference-time challenges.